In [1]:
using Lux, DiffEqFlux, OrdinaryDiffEq, Plots, Printf, Statistics
using ComponentArrays
using Optimization, OptimizationOptimisers
using Enzyme
using Random
using StaticArrays
using SciMLSensitivity
using SciMLStructures

Enzyme.API.looseTypeAnalysis!(true)

In [2]:
# Define the evolve! function
function evolve!(dc, c, p, t)
    dc .= c * p.p2 * p.p1
end

# Define the simulate function
function simulate(i1, i2, a, b, t_span)
    p2 = exp(-i2 * a)
    p1 = i1 * b
    p = (p1, p2)
    p_named = NamedTuple{(:p1, :p2)}(p)
    p = ComponentArray(p_named)
    c0 = [1.0 2.0; 1.0 0.0]
    prob = ODEProblem(evolve!, c0, t_span, p)
    sol = solve(prob, Tsit5())
    return Array(sol[end])
end

simulate (generic function with 1 method)

In [3]:
# Define the neural network and optimization process
rng = Xoshiro(0)
b = [0.0 1.0; 1.0 0.0]
a = 0.6
n = length(b[1, :])
i1 = 0.18
i2 = 2.5 
timespan = (0.0, 5.0)
p2 = exp(-i2 * a)
p1 = i1 * b
p = (p1, p2)
p_named = NamedTuple{(:p1, :p2)}(p)
p = ComponentArray(p_named)
c0 = [1.0 2.0; 1.0 0.0]
prob = ODEProblem(evolve!, c0, timespan, p)
sol = solve(prob, Euler(), dt = 0.5)
ans = Array(sol[end])

2×2 Matrix{Float64}:
 1.42176  2.23815
 1.01818  0.20179

In [4]:
inputs = [i1, i2]
input_size = length(inputs)
output_size = length(a) + length(b)
nn = Chain(
    Dense(input_size, input_size*3*n, tanh),
    Dense(input_size*3*n, output_size*2, tanh),
    Dense(output_size*2, output_size, sigmoid)
)

u, st = Lux.setup(rng, nn)

((layer_1 = (weight = Float32[-1.8019577 -1.1146251; -0.18273845 1.0075601; … ; 1.6416063 0.04306274; 1.0081401 -1.5636357], bias = Float32[-0.022002257, 0.5653541, 0.48737866, -0.109010376, 0.25698113, 0.29349717, -0.0496704, 0.5392088, 0.5833104, 0.61079556, -0.6089294, 0.47623715]), layer_2 = (weight = Float32[0.20824511 0.08736193 … 0.29591995 -0.76305753; -0.17639339 0.37737656 … -0.5897239 -0.32251218; … ; 0.52063733 -0.4213308 … -0.34941265 0.34940132; -0.40962756 0.7908495 … 0.2728175 -0.6145144], bias = Float32[-0.030394312, -0.11422878, 0.14924575, 0.082523964, -0.056747876, -0.24036361, -0.16483621, -0.10791249, -0.19515306, -0.023660526]), layer_3 = (weight = Float32[0.4909926 0.39306656 … -0.28983527 -0.15333983; -0.11188733 -0.5009294 … -0.2942955 0.279714; … ; 0.19163486 -0.22708948 … -0.20597327 0.05595271; -0.18737693 0.18280187 … 0.4284568 0.2179301], bias = Float32[-0.27912286, -0.2785868, 0.062313087, 0.078882255, -0.16884351])), (layer_1 = NamedTuple(), layer_2 = N

In [5]:
function predict_neuralode(u)
    output, outst = nn(inputs, u, st)
    p_a = output[1]
    pp_b = output[length(a)+1:end]
    p_b = zeros(n, n)
    index = 1
    for i in 1:n
        for j in 1:n
            p_b[i, j] = pp_b[index]
            index += 1
        end
    end
    nn_output = [p_a, p_b]
    pred = simulate(i1, i2, p_a, p_b, timespan)
    return Array(pred)
end

function loss_neuralode(u)
    pred = predict_neuralode(u)
    loss = sum(abs2, ans .- pred)
    return loss, pred
end

loss_neuralode (generic function with 1 method)

In [6]:
callback = function (state::Optimization.OptimizationState, loss_value::Float64; doplot = false)
    p = state.u
    l, pred = loss_neuralode(p)
    println(l)
    return false
end

#11 (generic function with 1 method)

In [7]:
pinit = ComponentArray(u)
adtype = Optimization.AutoEnzyme(; mode=set_runtime_activity(Reverse))
optf = Optimization.OptimizationFunction((x,_) -> loss_neuralode(x), adtype)
optprob = Optimization.OptimizationProblem(optf, pinit)

result_neuralode = Optimization.solve(
    optprob, OptimizationOptimisers.Adam(0.02); callback = callback, maxiters = 5)

┌ Warning: Mixed-Precision `matmul_cpu_fallback!` detected and Octavian.jl cannot be used for this set of inputs (C [Matrix{Float64}]: A [Base.ReshapedArray{Float32, 2, SubArray{Float32, 1, Vector{Float32}, Tuple{UnitRange{Int64}}, true}, Tuple{}}] x B [Matrix{Float64}]). Converting to common type to to attempt to use BLAS. This may be slow.
└ @ LuxLib.Impl /Users/posent/.julia/packages/LuxLib/kH9PB/src/impl/matmul.jl:148


0.11109365389306856
0.08153293262222544
0.05362805019263872
0.052768796223563626
0.03790652377516027
0.03790652377516027


retcode: Default
u: ComponentVector{Float32}(layer_1 = (weight = Float32[-1.8334342 -1.1461574; -0.22191949 0.96834075; … ; 1.5797973 -0.018746845; 1.0623716 -1.5093526], bias = Float32[-0.05352803, 0.52613914, 0.533315, -0.18827508, 0.18637878, 0.31784707, 0.0051922994, 0.53011614, 0.512857, 0.6893864, -0.67073894, 0.53051436]), layer_2 = (weight = Float32[0.25861672 0.036980662 … 0.34411877 -0.71267664; -0.10758178 0.30854985 … -0.52491826 -0.2536865; … ; 0.5945418 -0.49524552 … -0.27779365 0.42331484; -0.47451255 0.85574174 … 0.21080717 -0.67940766], bias = Float32[-0.08076895, -0.18304521, 0.08191717, 0.113361225, 0.019567706, -0.2899922, -0.1113793, -0.17244737, -0.26906088, 0.041226067]), layer_3 = (weight = Float32[0.41956013 0.3331338 … -0.23072289 -0.2130038; -0.039849557 -0.42566845 … -0.3696813 0.35499412; … ; 0.26461598 -0.14808464 … -0.28522068 0.13503328; -0.1488259 0.20106228 … 0.4109114 0.2360479], bias = Float32[-0.33880064, -0.20330976, 0.11330837, 0.15795779, -0.1507